In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

c:\Users\Camille\Documents\TWR


In [2]:
from app.config.container import campaign_service, request_service

c:\Users\Camille\Documents\TWR\deep_agents_twr\.venv\Lib\site-packages\motor\core.py:171: UserWarning: You appear to be connected to a DocumentDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/documentdb
  delegate = self.__delegate_class__(*args, **kwargs)


Garantindo índices...


In [3]:
traffic_sources = [
      # "google",
      "tiktok",
      # "taboola",
      # "facebook",
]

configs = [
      "fasttext",
]

In [ ]:
import pandas as pd
%reload_ext autoreload
%autoreload 2
from app.mil_attention.attention_mil import MILAttetionService

for source in traffic_sources:
      for emb_config in configs:

            hashes = await campaign_service.fetch_recent_active_campaigns(
                  traffic_source=source,
                  limit=100
            )

            requests = await request_service.fetch_training_sample_by_hashes(
                  hashes=hashes,
                  limit_each=10000
            )

            hashes_test = await campaign_service.fetch_recent_active_campaign_hashes_excluding(
                  excluded_hashes=hashes,
                  traffic_source=source
            )

            teste = await request_service.fetch_training_sample_by_hashes(
                  hashes=hashes_test,
                  limit_each=1000
            )

            data = pd.DataFrame(requests)
            data_teste = pd.DataFrame(teste)

            model_service = MILAttetionService(
                  traffic_source=source,
                  emb_config=emb_config
            )

            model_service.train(
                  data=data,
                  epochs=15
            )

            df_result = model_service.predict(data_teste)

            display(df_result)

Local file already updated: fasttext_tiktok.model
Local file already updated: fasttext_tiktok.model.syn1neg.npy
Local file already updated: fasttext_tiktok.model.wv.vectors_ngrams.npy
Modelo last modified no s3:  2026-03-02 20:06:34+00:00
Local file already updated
Enter to Fasttext encoder


Criando Vocabulário: 100%|██████████| 20000/20000 [00:28<00:00, 701.14it/s]


Using 11 out of 12 cores


Vetorizando: 100%|██████████| 20000/20000 [03:30<00:00, 94.85it/s] 


Finishing encoding


In [5]:
df_result["decision_mil"].value_counts()

decision_mil
1    1000
0    1000
Name: count, dtype: int64

In [6]:
df_result["mil_prediction"].value_counts()

mil_prediction
0    1481
1     519
Name: count, dtype: int64

In [7]:
from sklearn.metrics import classification_report

# 1. Comprime os dados de volta para 1 linha por Bag
bag_level_df = df_result.groupby('bag_id').agg(
    bag_true_label=('decision_mil', 'max'),     # Se tem pelo menos um bot (1), a bag é bot (1)
    bag_prediction=('mil_prediction', 'first')  # A previsão do modelo para aquela bag
).reset_index()

# 2. Cria a Matriz de Confusão
matriz_bag = pd.crosstab(
    bag_level_df['bag_true_label'], 
    bag_level_df['bag_prediction'], 
    rownames=['Real (Bag: 0=Segura, 1=Bot)'], 
    colnames=['Previsão do Modelo']
)

print("=== MATRIZ DE CONFUSÃO (NÍVEL DA BAG) ===\n")
display(matriz_bag)

# 3. Exibe as Métricas Avançadas (Precisão, Recall, F1-Score)
print("\n=== RELATÓRIO DE MÉTRICAS (NÍVEL DA BAG) ===")
print(classification_report(bag_level_df['bag_true_label'], bag_level_df['bag_prediction']))

=== MATRIZ DE CONFUSÃO (NÍVEL DA BAG) ===



Previsão do Modelo,0,1
"Real (Bag: 0=Segura, 1=Bot)",,
0,823,82
1,510,211



=== RELATÓRIO DE MÉTRICAS (NÍVEL DA BAG) ===
              precision    recall  f1-score   support

           0       0.62      0.91      0.74       905
           1       0.72      0.29      0.42       721

    accuracy                           0.64      1626
   macro avg       0.67      0.60      0.58      1626
weighted avg       0.66      0.64      0.59      1626



In [8]:
# Analisando as probabilidades brutas (antes de virarem 0 ou 1)
resumo_probabilidades = bag_level_df.merge(
    df_result[['bag_id', 'mil_bot_probability']].drop_duplicates(), 
    on='bag_id'
)

print("=== DISTRIBUIÇÃO DE PROBABILIDADE POR CLASSE REAL ===")
display(resumo_probabilidades.groupby('bag_true_label')['mil_bot_probability'].describe()[['mean', 'min', '50%', 'max']])

=== DISTRIBUIÇÃO DE PROBABILIDADE POR CLASSE REAL ===


,mean,min,50%,max
bag_true_label,,,,
0,0.373751,0.267188,0.362419,0.674481
1,0.455234,0.243704,0.412938,0.811573


In [9]:
# Supondo que o seu dataframe agrupado se chame df_result
# Vamos extrair a matemática de dentro das listas 'decision_mil'

# 1. Quantas requisições o IP fez no total? (Tamanho da Bag)
df_result['bag_size_real'] = df_result['decision_mil'].apply(len)

# 2. Quantas dessas requisições eram bots? (Soma dos 1s)
df_result['bot_requests_count'] = df_result['decision_mil'].apply(sum)

# 3. Qual a porcentagem de malícia na bag? (0.0 a 1.0)
df_result['bot_ratio'] = df_result['bot_requests_count'] / df_result['bag_size_real']

# 4. Classificação do Comportamento
def classificar_comportamento(ratio):
    if ratio == 0.0:
        return "100% Seguro (Apenas 0)"
    elif ratio == 1.0:
        return "100% Bot (Apenas 1)"
    else:
        return "Comportamento Misto (0 e 1)"

df_result['bag_behavior'] = df_result['bot_ratio'].apply(classificar_comportamento)

# ==========================================
# IMPRIMINDO O RELATÓRIO DE INSIGHTS
# ==========================================
print("=== INSIGHTS DO DATASET (NÍVEL DA BAG) ===\n")

print(f"Total de IPs únicos (Bags): {len(df_result)}")
print(f"Total de Requisições analisadas: {df_result['bag_size_real'].sum()}\n")

print("1. PERFIL DE COMPORTAMENTO DAS BAGS:")
print(df_result['bag_behavior'].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))
print("\n" + df_result['bag_behavior'].value_counts().to_string())

print("\n-------------------------------------------------")
print("2. VOLUME DE REQUISIÇÕES POR IP (Tamanho da Bag):")
print(df_result['bag_size_real'].describe()[['min', '50%', 'mean', 'max', 'std']])

print("\n-------------------------------------------------")
print("3. ANÁLISE DAS BAGS 'MISTAS' (A agulha no palheiro):")
mistos = df_result[df_result['bag_behavior'] == 'Comportamento Misto (0 e 1)']

if not mistos.empty:
    media_mistos = mistos['bot_ratio'].mean() * 100
    print(f"Quantidade de Bags Mistas: {len(mistos)}")
    print(f"Em média, {media_mistos:.1f}% das requisições num IP misto são de fato Bots.")
    print("Distribuição da % de malícia nos mistos:")
    print(mistos['bot_ratio'].describe()[['min', '50%', 'mean', 'max']])
else:
    print("Não existem bags mistas neste dataset.")

TypeError: object of type 'int' has no len()

In [ ]:
df_result["decision_mil"].value_counts()